In [1]:
import pandas as pd 
import numpy as np
import torch 
import torch.nn as nn
from torch.utils.data import Dataset , DataLoader
from torchmetrics import Accuracy,F1Score

In [2]:
data = pd.read_csv('dataset_pitt_cookie.csv')
data.head(2)
data = data[data['dx_label'].isin(['ProbableAD','Control'])]
data['dx_label'].unique()

array(['ProbableAD', 'Control'], dtype=object)

In [3]:
from sklearn.model_selection import train_test_split 
id_sujets = data['subject_id'].unique()
id_tr_vl , id_test = train_test_split ( id_sujets , test_size = 0.15 , random_state = 40 ) 
id_tr  , id_vl = train_test_split(id_tr_vl , test_size = 0.15 , random_state = 40 ) 
len(id_vl)

30

In [4]:
data_train = data[data['subject_id'].isin(id_tr)].reset_index()
data_val = data[data['subject_id'].isin(id_vl)].reset_index()
data_test = data[data['subject_id'].isin(id_test)].reset_index()

In [5]:
data_train.shape , data_val.shape , data_test.shape

((355, 22), (53, 22), (61, 22))

In [6]:
text_tr = data_train['transcript'].tolist()
tr_labels = (data_train['dx_label']=='ProbableAD').astype(int).tolist()
text_vl = data_val['transcript'].tolist()
val_labels = (data_val['dx_label']=='ProbableAD').astype(int).tolist()
text_test = data_test['transcript'].tolist()
test_labels = (data_test['dx_label']=='ProbableAD').astype(int).tolist()
text_tr[1]

"oh I see the sink is running over. I see the stool is tipping over. little boy's trying to get cookies out. the girl is reaching to get a cookie. the mother is drying dishes. the window's open. oh she's drying dishes now. she's standing in the water the cupboard door is open. there's a cookie jar. the brother's leaning against the cookie jar curtains on the window. there's some grass and some shrubbery outside the house next door. kitchen cabinets. oh I see three people in there, a mother and little boy and a girl."

In [7]:
from  transformers import RobertaTokenizer , RobertaModel
tokenizer  = RobertaTokenizer.from_pretrained('roberta-base')
encoding_tr = tokenizer(text_tr , padding = True , max_length = 256 , truncation  = True , return_tensors = 'pt')

In [8]:
encoding_tr = encoding_tr.to('cuda')
encoding_tr

C:\Users\youne\anaconda3\envs\torch_gpu\lib\site-packages\transformers\tokenization_utils_base.py:774: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  k: v.to(device=device, non_blocking=non_blocking) if hasattr(v, "to") and callable(v.to) else v


{'input_ids': tensor([[    0,   627,  1310,  ...,     1,     1,     1],
        [    0,  2678,    38,  ...,     1,     1,     1],
        [    0,   102,  2143,  ...,     1,     1,     1],
        ...,
        [    0,  3056,     6,  ...,     1,     1,     1],
        [    0,  3056,     6,  ...,     1,     1,     1],
        [    0,   119, 13492,  ...,     1,     1,     1]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]], device='cuda:0')}

In [9]:
class nlp_dataset ( Dataset):
    def __init__(self,text , labels , tokenizer ):
        self.text = text 
        self.labels = labels 
        self.tokenizer = tokenizer 
    def __len__(self):
        return len(self.labels)
    def __getitem__(self,idx):
        encoding = self.tokenizer(self.text[idx] , max_length = 256 , truncation = True , padding = 'max_length' , return_tensors = 'pt')
        return {
        'input_ids':encoding['input_ids'].squeeze(0),
        'attention_mask':encoding['attention_mask'].squeeze(0),
        'labels':torch.tensor(self.labels[idx] , dtype=torch.long) }

In [10]:
tr_nlp_dataset = nlp_dataset(text_tr,tr_labels,tokenizer)
val_nlp_dataset = nlp_dataset(text_vl,val_labels,tokenizer)
test_nlp_dataset = nlp_dataset(text_test,test_labels,tokenizer)

In [11]:
tr_loader = DataLoader(tr_nlp_dataset , shuffle = True , batch_size = 8 )
val_loader = DataLoader(val_nlp_dataset ,batch_size = 8 )
test_loader = DataLoader(test_nlp_dataset  , batch_size = 8 )

In [12]:
batch = next(iter(tr_loader))

print(batch['input_ids'].shape)
print(batch['attention_mask'].shape)
print(batch['labels'].shape)

torch.Size([8, 256])
torch.Size([8, 256])
torch.Size([8])


In [13]:
from transformers import RobertaForSequenceClassification
model = RobertaForSequenceClassification.from_pretrained('roberta-base' , num_labels = 2 ).to('cuda')

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [14]:
batch = next(iter(tr_loader))

In [15]:
output = model(batch['input_ids'].to('cuda') , attention_mask = batch['attention_mask'].to('cuda') , labels = batch['labels'].to('cuda'))
output

SequenceClassifierOutput(loss=tensor(0.6963, device='cuda:0', grad_fn=<NllLossBackward0>), logits=tensor([[-0.0166,  0.1535],
        [-0.0107,  0.1614],
        [-0.0155,  0.1711],
        [-0.0166,  0.1708],
        [-0.0194,  0.1756],
        [-0.0199,  0.1541],
        [-0.0299,  0.1701],
        [-0.0243,  0.1648]], device='cuda:0', grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)

In [16]:
print(batch['input_ids'].shape)
print(batch['attention_mask'].shape)

print(batch['input_ids'].dtype)
print(batch['attention_mask'].dtype)

torch.Size([8, 256])
torch.Size([8, 256])
torch.int64
torch.int64


In [17]:
model

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [18]:
optimizer = torch.optim.AdamW([
    {'params' :model.roberta.parameters() , 'lr':2e-5},
    {'params' : model.classifier.parameters() , 'lr':1e-4}],weight_decay=0.01)

In [19]:
from transformers import get_linear_schedule_with_warmup
num_training_steps = len(tr_loader)*3
print(str(len(tr_loader)))
print(str(num_training_steps))
scheduler = get_linear_schedule_with_warmup(
    optimizer , 
    num_warmup_steps = int(0.088 * num_training_steps),
    num_training_steps = num_training_steps)
print(str(int(0.088 * num_training_steps)))

45
135
11


In [20]:
from torchmetrics import Accuracy, F1Score, AUROC

def Train(model, train_loader, valid_loader, optimizer, epochs):
    metric_train_acc = Accuracy(task='binary').to('cuda')
    metric_valid_acc = Accuracy(task='binary').to('cuda')
    metric_train_auc = AUROC(task='binary').to('cuda')
    metric_valid_auc = AUROC(task='binary').to('cuda')
    metric_train_f1  = F1Score(task='binary').to('cuda')
    metric_valid_f1  = F1Score(task='binary').to('cuda')
    
    for epoch in range(epochs):
        metric_train_acc.reset()
        metric_valid_acc.reset()
        metric_train_auc.reset()
        metric_valid_auc.reset()
        metric_train_f1.reset()
        metric_valid_f1.reset()
        train_loss = 0
        valid_loss = 0
        
        model.train()
        for batch in train_loader:
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to('cuda')
            attention_mask = batch['attention_mask'].to('cuda')
            labels = batch['labels'].to('cuda')
            output = model(input_ids, attention_mask=attention_mask, labels=labels)
            logits = output.logits
            preds = torch.argmax(logits, dim=1)
            probs = torch.softmax(logits, dim=1)[:, 1]   
            metric_train_acc.update(preds, labels)
            metric_train_auc.update(probs, labels)
            metric_train_f1.update(preds, labels)
            loss = output.loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            train_loss += loss.item()
        
        model.eval()
        for batch2 in valid_loader:
            input_ids = batch2['input_ids'].to('cuda')
            attention_mask = batch2['attention_mask'].to('cuda')
            labels = batch2['labels'].to('cuda')
            with torch.no_grad():
                output = model(input_ids, attention_mask=attention_mask, labels=labels)
                logits = output.logits
                loss = output.loss
                valid_loss += loss.item()
                preds = torch.argmax(logits, dim=1)
                probs = torch.softmax(logits, dim=1)[:, 1]
                metric_valid_acc.update(preds, labels)
                metric_valid_auc.update(probs, labels)
                metric_valid_f1.update(preds, labels)
        
        print(f"epoch-------------{epoch}-------------")
        print(f"Train | Acc: {metric_train_acc.compute():.2f}  AUC: {metric_train_auc.compute():.2f}  F1: {metric_train_f1.compute():.2f}  Loss: {train_loss/len(train_loader):.4f}")
        print(f"Val   | Acc: {metric_valid_acc.compute():.2f}  AUC: {metric_valid_auc.compute():.2f}  F1: {metric_valid_f1.compute():.2f}  Loss: {valid_loss/len(valid_loader):.4f}")

In [21]:
Train(model , tr_loader ,val_loader,optimizer , 3)

epoch-------------0-------------
Train | Acc: 0.56  AUC: 0.60  F1: 0.70  Loss: 0.6878
Val   | Acc: 0.55  AUC: 0.81  F1: 0.33  Loss: 0.6820
epoch-------------1-------------
Train | Acc: 0.73  AUC: 0.81  F1: 0.73  Loss: 0.5311
Val   | Acc: 0.85  AUC: 0.93  F1: 0.85  Loss: 0.3978
epoch-------------2-------------
Train | Acc: 0.84  AUC: 0.90  F1: 0.84  Loss: 0.3747
Val   | Acc: 0.85  AUC: 0.95  F1: 0.85  Loss: 0.3793


In [27]:
torch.save(model.state_dict(),'best_roberta_seed_X_acc_0836.pt')